# Full TF expression quintiles and counts50 GRN recovery

Run from the package root. Full-data expression summaries and native Pearson scores come from the confirmed tables in `data/figure_inputs`. Network outcomes use the results recomputed by `02_network_topk.ipynb`. Outputs are written to `data/analysis_recomputed`, leaving the confirmed figure inputs unchanged. External expression matrices and full AUCell matrices are not required.

Reanalysis requested on 2026-11-06: split all **260 full-data TFs into five expression groups**, Q1 (lowest) to Q5 (highest), with 52 TFs per group. Sort by full mean CP10K, then TF name for ties. Save these fixed groups before joining recovery outcomes. Detection-rate quintiles are retained as a sensitivity analysis.

The full all-gene expression summary is reused from the executed `16_TF_expression_sensitivity.ipynb`. Its mean CP10K includes all 117,867 bins, including zeros. That notebook contains the original full-input CSR reading and expression calculation. This notebook reads the saved expression table, reconstructs full-target covariates and groups, and executes all group summaries and adjusted models. The earlier four-group analysis is outside this repository.

Use all 260 × 5 TF–repeat pairs. A missing full regulon has module recovery = 0 and target recovery = 0, with native AUCell Pearson unavailable. No new GRN inference or figures are generated. Reconstructed counts and five technical repeats retain the original data limitations.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display

BASE = Path.cwd()
RESULTS = BASE / "data/analysis_recomputed"
manifest = json.loads((BASE / "config.json").read_text())["analysis"]
full_regulons = json.loads(Path(manifest["full"]["regulons"]).read_text())
quintiles = ["Q1", "Q2", "Q3", "Q4", "Q5"]
display(pd.Series({"full_regulons": len(full_regulons), "repeats": len(manifest["runs"]), "full_input": manifest["full"]["input"]}))

full_regulons                                                  260
repeats                                                          5
full_input       /data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...
dtype: object

## Read fixed full-input expression

CP10K(b,g) = 10,000 × count(b,g) / total_count(b). The existing full-gene summary averages over all bins, including zero-expression bins, and supplies both TF expression and fixed full-target expression covariates. Expression is not redefined from any downsampled matrix.

In [2]:
gene_expression = pd.read_csv(BASE / "data/figure_inputs/full_gene_expression_summary.tsv", sep="\t", index_col="gene")
display(gene_expression.head(), pd.Series({
    "n_bins": manifest["n_bins"], "n_genes": len(gene_expression),
    "total_counts": gene_expression.total_counts.sum(),
    "sum_mean_cp10k": gene_expression.mean_cp10k.sum(),
}))

,total_counts,mean_counts,mean_cp10k,detection_rate
gene,,,,
A2M,15095,0.128068,0.591645,0.072429
A2ML1,548,0.004649,0.020926,0.003962
A3GALT2,261,0.002214,0.009115,0.001485
AAAS,4834,0.041012,0.177247,0.031366
AACS,29587,0.251020,1.045895,0.161589


n_bins               117867.0
n_genes               15541.0
total_counts      270918153.0
sum_mean_cp10k        10000.0
dtype: float64

## Lock full-data TF features and groups

Target covariates are arithmetic means of the full-input mean CP10K and detection rate across that TF's **full final targets, excluding the TF itself**. All modules have non-self targets. Recovery endpoints retain the existing full final edge definition, including self-edges. Full module size is the number of all full final targets.

The primary exposure for adjusted models is log1p(TF mean CP10K), standardized over the 260 TFs. Module size and target mean CP10K use log1p and the same standardization; detection rates are standardized without a log transform. Each fit uses either TF expression or TF detection rate, not both.

In [3]:
feature_rows = []
for regulon, targets in full_regulons.items():
    tf = regulon.rsplit("(", 1)[0]
    target_genes = sorted(set(targets) - {tf})
    expression = gene_expression.loc[tf]
    target_expression = gene_expression.loc[target_genes]
    feature_rows.append({
        "regulon": regulon, "TF": tf,
        "tf_mean_cp10k": expression.mean_cp10k,
        "tf_detection_rate": expression.detection_rate,
        "n_targets_full": len(targets),
        "n_nonself_targets": len(target_genes),
        "target_mean_cp10k": target_expression.mean_cp10k.mean(),
        "target_detection_rate": target_expression.detection_rate.mean(),
    })
features = pd.DataFrame(feature_rows).sort_values(["tf_mean_cp10k", "TF"]).reset_index(drop=True)
features["expression_rank"] = np.arange(1, len(features) + 1)
features["expression_quintile"] = pd.qcut(features.expression_rank, 5, labels=quintiles)
detection_order = features.sort_values(["tf_detection_rate", "TF"]).index
features.loc[detection_order, "detection_rank"] = np.arange(1, len(features) + 1)
features["detection_quintile"] = pd.qcut(features.detection_rank, 5, labels=quintiles)
for column in ["tf_mean_cp10k", "n_targets_full", "target_mean_cp10k"]:
    values = np.log1p(features[column])
    features["z_log_" + column] = (values - values.mean()) / values.std(ddof=0)
for column in ["tf_detection_rate", "target_detection_rate"]:
    values = features[column]
    features["z_" + column] = (values - values.mean()) / values.std(ddof=0)
features.to_csv(RESULTS / "tf_expression_quintile_features.tsv", sep="\t", index=False)
display(features.groupby("expression_quintile", observed=True).agg(
    n_TFs=("TF", "size"), min_cp10k=("tf_mean_cp10k", "min"),
    median_cp10k=("tf_mean_cp10k", "median"), max_cp10k=("tf_mean_cp10k", "max"),
    median_detection=("tf_detection_rate", "median"), median_full_targets=("n_targets_full", "median"),
))

,n_TFs,min_cp10k,median_cp10k,max_cp10k,median_detection,median_full_targets
expression_quintile,,,,,,
Q1,52,0.000229,0.062536,0.101402,0.009540,19.5
Q2,52,0.108426,0.149443,0.202226,0.021685,32.5
Q3,52,0.204217,0.260890,0.333062,0.040754,183.5
Q4,52,0.335901,0.469016,0.618243,0.070155,206.5
Q5,52,0.628677,1.150971,6.407020,0.155226,307.5


## Recovery outcomes: all 260 × 5 pairs

A module is recovered when the final signed regulon name is retained. Target recovery is shared full targets / all full targets. The **edge-weighted** group rate pools edges; the **TF-average** rate gives equal weight to each TF and includes missing modules as zero. Pearson uses each run's own inferred regulon AUCell scores across paired bins and is conditional on module recovery; its coverage is reported explicitly.

In [4]:
network = pd.read_csv(RESULTS / "network_regulons.tsv", sep="\t")
pearson = pd.read_csv(BASE / "data/figure_inputs/native_shared_regulon_pearson.tsv", sep="\t")
columns = ["run", "seed", "regulon", "status", "n_targets_sample", "shared_targets", "target_retention", "target_jaccard"]
endpoints = network.loc[network.regulon.isin(full_regulons), columns].merge(features, on="regulon", validate="many_to_one")
endpoints["module_recovered"] = (endpoints.status == "retained").astype(int)
endpoints = endpoints.merge(pearson[["run", "regulon", "pearson"]], on=["run", "regulon"], how="left", validate="one_to_one")
endpoints.to_csv(RESULTS / "tf_expression_quintile_endpoints.tsv", sep="\t", index=False)
summary_rows = []
for grouping in ["expression_quintile", "detection_quintile"]:
    for (run, quintile), group in endpoints.groupby(["run", grouping], observed=True):
        scores = group.pearson.dropna()
        summary_rows.append({
            "grouping": grouping, "run": run, "quintile": quintile,
            "n_full_TFs": len(group), "n_recovered_modules": group.module_recovered.sum(),
            "module_recovery": group.module_recovered.mean(),
            "full_edges": group.n_targets_full.sum(), "shared_edges": group.shared_targets.sum(),
            "edge_weighted_recovery": group.shared_targets.sum() / group.n_targets_full.sum(),
            "TF_mean_target_recovery": group.target_retention.mean(),
            "TF_median_target_recovery": group.target_retention.median(),
            "n_Pearson": len(scores), "Pearson_coverage": len(scores) / len(group),
            "Pearson_mean": scores.mean(), "Pearson_median": scores.median(),
            "Pearson_q25": scores.quantile(.25), "Pearson_q75": scores.quantile(.75),
        })
summary = pd.DataFrame(summary_rows).sort_values(["grouping", "run", "quintile"])
summary.to_csv(RESULTS / "tf_expression_quintile_group_runs.tsv", sep="\t", index=False)
metrics = ["module_recovery", "edge_weighted_recovery", "TF_mean_target_recovery", "Pearson_median", "Pearson_coverage"]
across_repeats = summary.groupby(["grouping", "quintile"])[metrics].agg(["mean", "std", "min", "max"])
across_repeats.to_csv(RESULTS / "tf_expression_quintile_group_summary.tsv", sep="\t")
display(summary.loc[summary.grouping == "expression_quintile"], across_repeats.loc["expression_quintile"])


,grouping,run,quintile,n_full_TFs,n_recovered_modules,module_recovery,full_edges,shared_edges,edge_weighted_recovery,TF_mean_target_recovery,TF_median_target_recovery,n_Pearson,Pearson_coverage,Pearson_mean,Pearson_median,Pearson_q25,Pearson_q75
0,expression_quintile,rep01,Q1,52,14,0.269231,1732,144,0.083141,0.051564,0.000000,14,0.269231,0.464004,0.283441,0.207592,0.752260
1,expression_quintile,rep01,Q2,52,29,0.557692,3263,598,0.183267,0.140471,0.034884,29,0.557692,0.654913,0.779789,0.476131,0.827317
2,expression_quintile,rep01,Q3,52,45,0.865385,12803,2967,0.231743,0.247286,0.220350,45,0.865385,0.780308,0.865262,0.789006,0.920125
3,expression_quintile,rep01,Q4,52,49,0.942308,17972,4702,0.261629,0.355106,0.316015,49,0.942308,0.830300,0.865227,0.778845,0.909589
4,expression_quintile,rep01,Q5,52,46,0.884615,32722,12925,0.394994,0.418898,0.392773,46,0.884615,0.783892,0.859083,0.780620,0.903682
5,expression_quintile,rep02,Q1,52,16,0.307692,1732,206,0.118938,0.084442,0.000000,16,0.307692,0.606654,0.695287,0.436346,0.801620
6,expression_quintile,rep02,Q2,52,28,0.538462,3263,562,0.172234,0.127288,0.000000,28,0.538462,0.686827,0.773169,0.585389,0.857691
7,expression_quintile,rep02,Q3,52,43,0.826923,12803,2876,0.224635,0.223011,0.200906,43,0.826923,0.804473,0.876662,0.796631,0.906328
8,expression_quintile,rep02,Q4,52,47,0.903846,17972,4588,0.255286,0.351693,0.321703,47,0.903846,0.819175,0.869042,0.783447,0.905045
9,expression_quintile,rep02,Q5,52,47,0.903846,32722,13161,0.402206,0.425147,0.427916,47,0.903846,0.788765,0.861392,0.743585,0.913497


module_recovery                               edge_weighted_recovery  \
                    mean       std       min       max                   mean   
quintile                                                                        
Q1              0.265385  0.036991  0.211538  0.307692               0.095843   
Q2              0.530769  0.032179  0.480769  0.557692               0.179589   
Q3              0.846154  0.038462  0.807692  0.903846               0.226994   
Q4              0.919231  0.016090  0.903846  0.942308               0.258669   
Q5              0.892308  0.017201  0.865385  0.903846               0.394805   

                                       TF_mean_target_recovery            \
               std       min       max                    mean       std   
quintile                                                                   
Q1        0.015433  0.083141  0.118938                0.070870  0.014957   
Q2        0.008429  0.171008  0.191542                0.130808  0.010543   
Q3        0.005582  0.221042  0.234008                0.244105  0.013677   
Q4        0.007083  0.251447  0.269530                0.350107  0.003242   
Q5        0.004837  0.389402  0.402206                0.410445  0.012989   

                             Pearson_median                                \
               min       max           mean       std       min       max   
quintile                                                                    
Q1        0.051564  0.085919       0.642156  0.202320  0.283441  0.767890   
Q2        0.117333  0.142549       0.772482  0.026420  0.727471  0.794670   
Q3        0.223011  0.258124       0.869993  0.005552  0.863481  0.876662   
Q4        0.347730  0.355106       0.862769  0.010202  0.849143  0.874525   
Q5        0.394882  0.425147       0.853177  0.016330  0.824726  0.865453   

         Pearson_coverage                                
                     mean       std       min       max  
quintile                                                 
Q1               0.265385  0.036991  0.211538  0.307692  
Q2               0.530769  0.032179  0.480769  0.557692  
Q3               0.846154  0.038462  0.807692  0.903846  
Q4               0.919231  0.016090  0.903846  0.942308  
Q5               0.892308  0.017201  0.865385  0.903846

## Exploratory adjusted associations

Fit two endpoints for each exposure definition: binary module recovery and fractional target recovery. Both use a binomial-logit GEE with independent working correlation, robust covariance clustered by TF (260 clusters), and repeat fixed effects. Fractional target recovery is **unweighted across TFs**: it estimates the TF-average association rather than allowing large modules to dominate through their edge counts. Missing modules remain zero in both models.

Adjustment: full log module size, log target mean CP10K, and target detection rate. The expression exposure is the primary analysis; TF detection is the predefined sensitivity analysis. ORs describe a one-SD increase on the modeled exposure scale and are associations, not causal effects. For fractional recovery, OR refers to the modeled odds of the mean recovery proportion. Pearson is described conditionally, not modeled as a full-cohort endpoint.

Maximum 100 iterations per fit. Report convergence, design condition number, and covariate correlations. The two primary exposure p values receive a Holm correction; sensitivity p values remain exploratory. These uncertainty estimates concern TF-level association within this input and do not measure biological between-donor variability.

These models use continuous expression/detection exposures rather than quintile membership. Changing four groups to five changes the descriptive group results but does not change the model specification or its TF–repeat observations; the models are re-executed here for a complete record.

In [5]:
covariates = ["z_log_n_targets_full", "z_log_target_mean_cp10k", "z_target_detection_rate"]
exposures = {"expression": "z_log_tf_mean_cp10k", "detection": "z_tf_detection_rate"}
model_rows = []
diagnostics = []
for exposure_name, exposure in exposures.items():
    for outcome in ["module_recovered", "target_retention"]:
        formula = outcome + " ~ " + " + ".join([exposure] + covariates + ["C(run)"])
        model = smf.gee(formula, groups="TF", data=endpoints,
                        family=sm.families.Binomial(), cov_struct=sm.cov_struct.Independence())
        fit = model.fit(maxiter=100, cov_type="robust")
        intervals = fit.conf_int()
        for term in fit.params.index:
            model_rows.append({
                "exposure": exposure_name, "outcome": outcome, "term": term,
                "coefficient": fit.params[term], "SE": fit.bse[term], "p_value": fit.pvalues[term],
                "odds_ratio": np.exp(fit.params[term]),
                "OR_CI_low": np.exp(intervals.loc[term, 0]), "OR_CI_high": np.exp(intervals.loc[term, 1]),
            })
        diagnostics.append({
            "exposure": exposure_name, "outcome": outcome, "converged": fit.converged,
            "n_observations": int(fit.nobs), "n_TF_clusters": model.num_group,
            "design_condition_number": np.linalg.cond(model.exog),
        })
models = pd.DataFrame(model_rows)
model_diagnostics = pd.DataFrame(diagnostics)
models.to_csv(RESULTS / "tf_expression_quintile_model_coefficients.tsv", sep="\t", index=False)
model_diagnostics.to_csv(RESULTS / "tf_expression_quintile_model_diagnostics.tsv", sep="\t", index=False)
exposure_effects = models.loc[[row.term == exposures[row.exposure] for row in models.itertuples()]].copy()
from statsmodels.stats.multitest import multipletests
primary = exposure_effects.exposure == "expression"
exposure_effects.loc[primary, "p_Holm_primary"] = multipletests(exposure_effects.loc[primary, "p_value"], method="holm")[1]
exposure_effects.to_csv(RESULTS / "tf_expression_quintile_exposure_effects.tsv", sep="\t", index=False)
correlations = features[list(exposures.values()) + covariates].corr()
correlations.to_csv(RESULTS / "tf_expression_quintile_covariate_correlations.tsv", sep="\t")
display(exposure_effects, model_diagnostics, correlations)


,exposure,outcome,term,coefficient,SE,p_value,odds_ratio,OR_CI_low,OR_CI_high,p_Holm_primary
5,expression,module_recovered,z_log_tf_mean_cp10k,1.178781,0.413200,4.333466e-03,3.250408,1.446177,7.305573,4.333466e-03
14,expression,target_retention,z_log_tf_mean_cp10k,0.532163,0.081483,6.535739e-11,1.702611,1.451298,1.997441,1.307148e-10
23,detection,module_recovered,z_tf_detection_rate,1.261375,0.554664,2.295846e-02,3.530271,1.190358,10.469805,NaN
32,detection,target_retention,z_tf_detection_rate,0.462389,0.078713,4.245126e-09,1.587863,1.360856,1.852738,NaN


,exposure,outcome,converged,n_observations,n_TF_clusters,design_condition_number
0,expression,module_recovered,True,1300,260,11.429411
1,expression,target_retention,True,1300,260,11.429411
2,detection,module_recovered,True,1300,260,11.418221
3,detection,target_retention,True,1300,260,11.418221


,z_log_tf_mean_cp10k,z_tf_detection_rate,z_log_n_targets_full,z_log_target_mean_cp10k,z_target_detection_rate
z_log_tf_mean_cp10k,1.000000,0.983411,0.437816,-0.294713,-0.284952
z_tf_detection_rate,0.983411,1.000000,0.427581,-0.270351,-0.253735
z_log_n_targets_full,0.437816,0.427581,1.000000,-0.519577,-0.529719
z_log_target_mean_cp10k,-0.294713,-0.270351,-0.519577,1.000000,0.980012
z_target_detection_rate,-0.284952,-0.253735,-0.529719,0.980012,1.000000


## Collinearity diagnostic and additional sensitivity fits

The two full-target covariates are strongly correlated (r ≈ 0.98). This diagnostic motivated **additional, post hoc sensitivity fits** in the original four-group analysis; they are retained here, not a change to the primary model or the expression groups. Report variance inflation factors on the 260 unique full TFs, including an intercept. For the primary TF expression exposure, fit each endpoint with module size and either target mean CP10K or target detection rate. Retain the original model and all alternatives; do not select a model by its p value.

There are four additional fits, each with the same TF clustering, repeat fixed effects, robust covariance, and maximum 100 iterations. These checks assess whether the TF-expression association depends on including both highly correlated target covariates.

In [6]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_rows = []
for exposure_name, exposure in exposures.items():
    design = sm.add_constant(features[[exposure] + covariates])
    for index, term in enumerate(design.columns[1:], start=1):
        vif_rows.append({"exposure": exposure_name, "term": term,
                         "VIF": variance_inflation_factor(design.to_numpy(), index)})
vif = pd.DataFrame(vif_rows)
vif.to_csv(RESULTS / "tf_expression_quintile_covariate_VIF.tsv", sep="\t", index=False)

sensitivity_rows = []
for adjustment, target_covariate in {
    "target_expression_only": "z_log_target_mean_cp10k",
    "target_detection_only": "z_target_detection_rate",
}.items():
    for outcome in ["module_recovered", "target_retention"]:
        formula = outcome + " ~ z_log_tf_mean_cp10k + z_log_n_targets_full + " + target_covariate + " + C(run)"
        model = smf.gee(formula, groups="TF", data=endpoints,
                        family=sm.families.Binomial(), cov_struct=sm.cov_struct.Independence())
        fit = model.fit(maxiter=100, cov_type="robust")
        term = "z_log_tf_mean_cp10k"
        interval = fit.conf_int().loc[term]
        sensitivity_rows.append({
            "adjustment": adjustment, "outcome": outcome,
            "coefficient": fit.params[term], "SE": fit.bse[term], "p_value": fit.pvalues[term],
            "odds_ratio": np.exp(fit.params[term]),
            "OR_CI_low": np.exp(interval.iloc[0]), "OR_CI_high": np.exp(interval.iloc[1]),
            "converged": fit.converged, "n_observations": int(fit.nobs),
            "n_TF_clusters": model.num_group, "design_condition_number": np.linalg.cond(model.exog),
        })
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(RESULTS / "tf_expression_quintile_collinearity_sensitivity.tsv", sep="\t", index=False)
display(vif, sensitivity)


,exposure,term,VIF
0,expression,z_log_tf_mean_cp10k,1.252335
1,expression,z_log_n_targets_full,1.588963
2,expression,z_log_target_mean_cp10k,25.454535
3,expression,z_target_detection_rate,25.757794
4,detection,z_tf_detection_rate,1.243074
5,detection,z_log_n_targets_full,1.596166
6,detection,z_log_target_mean_cp10k,25.634936
7,detection,z_target_detection_rate,25.945641


,adjustment,outcome,coefficient,SE,p_value,odds_ratio,OR_CI_low,OR_CI_high,converged,n_observations,n_TF_clusters,design_condition_number
0,target_expression_only,module_recovered,1.199950,0.408851,3.336178e-03,3.319952,1.489764,7.398541,True,1300,260,7.325353
1,target_expression_only,target_retention,0.556026,0.085080,6.347641e-11,1.743730,1.475906,2.060154,True,1300,260,7.325353
2,target_detection_only,module_recovered,1.198529,0.405608,3.127731e-03,3.315237,1.497134,7.341224,True,1300,260,7.328238
3,target_detection_only,target_retention,0.556098,0.084038,3.660921e-11,1.743854,1.479029,2.056098,True,1300,260,7.328238


## High versus low expression and proposed practical thresholds

Q5 − Q1 is descriptive for each technical repeat. The plan proposed exploratory thresholds of 90% module recovery, 80% edge recovery, and median shared-module Pearson 0.90, each checked in all five repeats. They are **not universal biological standards or a formal equivalence test**. Relative improvement and small absolute loss are separate conclusions; a high conditional Pearson does not recover missing modules.

In [7]:
contrast_rows = []
threshold_rows = []
for grouping in ["expression_quintile", "detection_quintile"]:
    selected = summary.loc[summary.grouping == grouping]
    for run in sorted(selected.run.unique()):
        group = selected.loc[selected.run == run].set_index("quintile")
        contrast_rows.append({
            "grouping": grouping, "run": run,
            **{metric + "_Q5_minus_Q1": group.loc["Q5", metric] - group.loc["Q1", metric] for metric in metrics},
        })
    high = selected.loc[selected.quintile == "Q5"]
    for metric, threshold in [("module_recovery", .90), ("edge_weighted_recovery", .80), ("Pearson_median", .90)]:
        threshold_rows.append({
            "grouping": grouping, "metric": metric, "threshold": threshold,
            "Q5_mean": high[metric].mean(), "Q5_min": high[metric].min(), "Q5_max": high[metric].max(),
            "repeats_meeting_threshold": (high[metric] >= threshold).sum(), "n_repeats": len(high),
        })
contrasts = pd.DataFrame(contrast_rows)
thresholds = pd.DataFrame(threshold_rows)
contrasts.to_csv(RESULTS / "tf_expression_quintile_Q5_vs_Q1.tsv", sep="\t", index=False)
thresholds.to_csv(RESULTS / "tf_expression_quintile_practical_thresholds.tsv", sep="\t", index=False)
display(contrasts, thresholds)


,grouping,run,module_recovery_Q5_minus_Q1,edge_weighted_recovery_Q5_minus_Q1,TF_mean_target_recovery_Q5_minus_Q1,Pearson_median_Q5_minus_Q1,Pearson_coverage_Q5_minus_Q1
0,expression_quintile,rep01,0.615385,0.311853,0.367334,0.575642,0.615385
1,expression_quintile,rep02,0.596154,0.283269,0.340705,0.166106,0.596154
2,expression_quintile,rep03,0.576923,0.287312,0.308963,0.122132,0.576923
3,expression_quintile,rep04,0.692308,0.303951,0.342225,0.134390,0.692308
4,expression_quintile,rep05,0.653846,0.308423,0.338645,0.056836,0.653846
5,detection_quintile,rep01,0.596154,0.283209,0.373657,0.624024,0.596154
6,detection_quintile,rep02,0.576923,0.261789,0.355254,0.222111,0.576923
7,detection_quintile,rep03,0.538462,0.273633,0.334133,0.406807,0.538462
8,detection_quintile,rep04,0.673077,0.290650,0.352166,0.171399,0.673077
9,detection_quintile,rep05,0.615385,0.262964,0.331572,0.232256,0.615385


,grouping,metric,threshold,Q5_mean,Q5_min,Q5_max,repeats_meeting_threshold,n_repeats
0,expression_quintile,module_recovery,0.9,0.892308,0.865385,0.903846,3,5
1,expression_quintile,edge_weighted_recovery,0.8,0.394805,0.389402,0.402206,0,5
2,expression_quintile,Pearson_median,0.9,0.853177,0.824726,0.865453,0,5
3,detection_quintile,module_recovery,0.9,0.869231,0.826923,0.884615,0,5
4,detection_quintile,edge_weighted_recovery,0.8,0.372648,0.367368,0.379201,0,5
5,detection_quintile,Pearson_median,0.9,0.855386,0.825906,0.870297,0,5
